# P5 — Granite na própria LinuxONE

Este notebook roda **na VM da equipe** (s390x). Ele:

1. confere o que a VM oferece (o kernel da VM já tem o `llama_cpp`, o llama-cpp-python);
2. carrega o `granite-4.0-h-350m-base-be.f16.gguf` **dentro deste kernel**, sem servidor;
3. gera explicações de alertas com esse Granite e mostra o que a validação aceitou ou recusou.

Sobre o arquivo: GGUF é o formato do llama.cpp; `-be` = big-endian (obrigatório no s390x);
`f16` = pesos em 16 bits (~685 MB na memória); `h-350m` = Granite 4.0 híbrido de 350 milhões de
parâmetros; `base` = **não** é ajustado para seguir instruções. Por isso o prompt é por exemplos
(few-shot), e todo texto passa pela mesma validação: se inventar algo, volta o texto padrão.

**A VM é compartilhada (3 vCPUs e ~8 GB para a equipe toda):** 1 thread; rode a última célula
(`close`) e desligue o kernel (aba *Running*) ao terminar. Os alertas aqui são **mock**.

In [ ]:
import os
import sys
import time
from pathlib import Path

# Sobe a partir da pasta do notebook até o clone (ex.: ~/shared/civicwater/users/<nome>/repo).
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(REPO / "src"))

for v in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"]:
    os.environ[v] = "1"

os.environ.setdefault("LOCAL_LLM_MODEL_PATH", str(Path.home() / "granite-4.0-h-350m-base-be.f16.gguf"))
os.environ.setdefault("LOCAL_LLM_THREADS", "1")

print("repo:", REPO)
print("python do kernel:", sys.executable, sys.version.split()[0])
print("modelo:", os.environ["LOCAL_LLM_MODEL_PATH"])

## 1. O que esta VM tem (só leitura)

In [ ]:
import glob
import importlib.util
import platform
import shutil
import struct

home = Path.home()
print("arquitetura:", platform.machine(), "| CPUs:", os.cpu_count())
try:
    meminfo = Path("/proc/meminfo").read_text().splitlines()
    print(*[l for l in meminfo if l.startswith(("MemTotal", "MemAvailable"))], sep="\n")
    cpuinfo = Path("/proc/cpuinfo").read_text()
    print("NNPA (acelerador de IA Telum) visível:", " nnpa" in cpuinfo or "nnpa " in cpuinfo)
except OSError:
    pass

print("\nprogramas:")
for b in ["llama-server", "llama-cli", "gcc", "cmake", "uv"]:
    print(f"  {b:13} {shutil.which(b)}")

print("\npacotes Python neste kernel:")
for m in ["llama_cpp", "pydantic", "fastapi", "pytest"]:
    print(f"  {m:13} {'sim' if importlib.util.find_spec(m) else '-'}")

print("\narquivos .gguf:")
for p in sorted(set(glob.glob(str(home / "*.gguf")) + glob.glob(str(home / "*/*.gguf")))):
    with open(p, "rb") as f:
        head = f.read(8)
    if head[:4] != b"GGUF":
        print("  ", p, "-> não parece GGUF")
        continue
    le, be = struct.unpack("<I", head[4:]), struct.unpack(">I", head[4:])
    endian = "big-endian" if be[0] < le[0] else "little-endian"
    print(f"   {p}  ({os.path.getsize(p) / 1e6:.0f} MB, GGUF v{min(le[0], be[0])}, {endian})")

## 2. Carregar o Granite neste kernel

Carrega uma vez (alguns segundos) e fica na memória até a última célula (`close`).
A linha `system info` mostra o que o llama.cpp usa na CPU (procure `VXE`, e `NNPA`/`ZDNN` se o
build tiver o acelerador). Se der erro aqui, copie a mensagem: pode ser a versão do
llama-cpp-python sem suporte ao Granite 4.0 híbrido (`granitehybrid`).

In [ ]:
import llama_cpp

from sentinel.narrative import LlamaCppNarrator, LocalLLMError

print("llama-cpp-python:", getattr(llama_cpp, "__version__", "?"))
try:
    info = llama_cpp.llama_print_system_info()
    print("system info:", info.decode() if isinstance(info, bytes) else info)
except Exception as exc:
    print("system info indisponível:", exc)

granite = LlamaCppNarrator.from_env()
t = time.perf_counter()
try:
    granite.load()
    print(f"modelo carregado em {time.perf_counter() - t:.1f} s:", granite.name)
except LocalLLMError as exc:
    print("NÃO carregou:", exc)

## 3. Explicar um alerta com o Granite local

`explain()` manda ao modelo dois exemplos inventados e os fatos do alerta. O texto só é aceito se
**todo** sensor, região, número, data e horário estiver no alerta; senão volta o texto padrão,
com o motivo em `fallback_reason`.

In [ ]:
from sentinel.narrative import explain
from sentinel.narrative.local import build_completion_prompt
from sentinel.narrative.mock import mock_alert

alerta = mock_alert()  # inventado: região R4, CUSUM, 3 sensores
print("...", build_completion_prompt(alerta)[-500:], "\n")

t = time.perf_counter()
e = explain(alerta, granite)
print(f"tempo: {time.perf_counter() - t:.1f} s | fonte: {e.source} | fallback: {e.fallback_reason}")
print("validação:", e.check)
print("\n" + e.text)

## 4. Mesmo teste com um alerta do pipeline (detector mock do backend)

In [ ]:
from datetime import UTC, datetime

from sentinel.audit import HashChainAuditSink
from sentinel.config import Settings
from sentinel.main import build_pipeline
from sentinel.schemas.windows import SensorWindow
from sentinel.sensors import SensorRegistry
from sentinel.synthetic import generate

registry = SensorRegistry.load(REPO / "config" / "sensors.provisional.json")
pipeline = build_pipeline(Settings(), registry=registry, audit_sink=HashChainAuditSink())
for w in generate(datetime(2019, 3, 14, tzinfo=UTC), 288, registry, seed=7, noise_m=0.05,
                  leak_at=200, leak_sensors=registry.pressure[:3]):
    pipeline.ingest(SensorWindow.model_validate(w))

alerta_pipeline = pipeline.alerts.query(limit=1, offset=0)[0][0]
for idioma in ("en", "pt"):
    t = time.perf_counter()
    e = explain(alerta_pipeline, granite, language=idioma)
    print(f"[{idioma}] tempo: {time.perf_counter() - t:.1f} s | fonte: {e.source} | fallback: {e.fallback_reason}")
    print("validação:", e.check)
    print(e.text, "\n")

## 5. Como ler o resultado

- `fonte: llm` → o texto do Granite passou na validação e seria mostrado ao operador.
- `fonte: template` com `narrative_rejected` → o modelo inventou ou copiou algo (veja `validação`);
  o operador vê o texto padrão. Com um modelo *base* de 350M isso pode ser frequente.
- `narrator_error` → o modelo não carregou ou falhou (mensagem em `fallback_reason`).

Anote o **tempo de carga** (seção 2) e o **tempo por explicação** (seções 3 e 4): eles decidem
se a explicação pode ser gerada quando o operador abre o alerta ou se deve ser gerada antes, em
segundo plano. Se a taxa de aceitação for baixa, as alternativas são um Granite *instruct*
convertido para big-endian ou o `ibm/granite-4-h-small` no watsonx (`WatsonxNarrator`).

Pela linha de comando (terminal, depois do `source …/start.sh`):
`python -m sentinel.narrative --llamacpp` (mostra `model_load_s` e `elapsed_s`).

## 6. Liberar a memória

In [ ]:
granite.close()
print("modelo descarregado:", not granite.loaded)